# Threading
**1. Python Concurrency**
* A *process* is a computer program.
* A *thread* refers to a thread of execution within a computer program.

Each program is a process that has at least one thread that executes instructions for that process.

* *Concurrency* refers to executing tasks out of order.
* *Parallelism* refers to executing tasks simultaneously.

When we are developing code, we can achieve concurrency with or without parallelism, although concurrency (e.g., task order being irrelevant) is a prerequesite for parallelism.
Our goal is to speed-up a program by executing two or more tasks simultaneously. We are almost always interested in paralellism when we talk about Python concurrency.

**2. Thread**

Thread-base concurrency is provided via the `threading` module and the `Thread` class.

When to use it? Use it for blocking IO-bound tasks which involve calling instructions in our operating system (the kernel), which will wait for the operation to complete.
* Reading or writing a file from the hard drive.
* Reading or writing to standard output, input, or error (`stdin`, `stdout`, `stderr`).
* Printing a document.
* Socket connections.
* Downloading or uploading a file.
* Querying a server.
* Querying a database.
* Taking a photo or recording a video.

Exception: One execption are those tasks that execute code in a third-party library that explicitly releases GIL.
* When calculating cryptographic hashes using the `hashlib` library.
* When calculating matrix operations using the `Numpy` library from the `SciPy` suite.
* Whe compressing data or files (e.g., `zlib`) and when working with video and image data (e.g., `OpenCV`).


Limitations: A key limitation of `Thread` for thread-based concurrency is that it is subject to the Global Intepreter Lock (GIL). This means that only one thread can run at a time in a Python process, unless the GIL is released, such as during blocking I/O or explicitly in a third-party libraries. This limitation means that although threads can achieve concurrency (executing tasks out of order), it can only achieve parallelism (executing tasks simultaneously) under specific circumstances.

**3. Processes**

Process-based concurrency is provided via the `multiprocessing` module and the `Process` class.

When to use it? Use it for CPU-bound tasks which involve performing a computation and does not involve IO, and where data needs to be shared between processes. They typically only involve data in main memory (RAM) or cache and performing computations on or with that data. 
* Calculating points in a fractal.
* Estimating PI.
* Facoting primes.
* Parsing HTML, JSON, etc. documents.
* Processing text.
* Running simulations.

Limitations: A key limitation of the `multiprocessing` module is sharing data between processes. Unlike threads that have shared memory, processes must share data and variables using Inter-Process Communitation (IPC). This requires that data be serialized in order to be transmitted to another process, and deserialized once received. Python-native serialization in the `pickle` module is used and performed automatically. Data is shared using pipes or queues.

Why note always use `multiprocessing`?
1. Heavyweight: Processes are heavyweight structures making them slower to start and require more memory.
2. IPC: All data sent between processes must be serialized, adding a computational overhead proportional to the data shared.
3. Limited Number: The operating system imposes limits on the number of child processes we can create.

**4. Global Interpreter Lock (GIL)**

Python uses GIL to make instructions executed by the Python interpreter thread-safe. It uses a synchronization primitive called a mutual exclusion or a mutex lock to ensure that only one thread of execution can execute instructions at a time within a Python process.

The effect of the GIL is that whenever a thread within a Python program wants to run, it must acquire the lock before executing. This is not a problem for most python programs that have a single thread of execution, called the main thread. It can become a problem in multi-headed Python programs.

The lock is also released in certain situations, allowing other threads to execute. This can happen, for example, during blocking I/O operations or when third-party Python libraries perform computationally intensive tasks in C code (e.g., array operations in NumPy).

## Create & Start New Threads
* *Main Thread*: Default thread created by a main process in Python program, has the name *MainThread*. 
* *Helper Threads*: There may be other threads in our program, created by the standard library or third-party libraries, that assist a primary activity.
* *New Threads*: New threads created directly in our program.
* *Worker Threads*: We may create a pool of reusable worker threads, called a thread pool. We do not create threads in the pool directly, they are created for us by the pool. They are a specific type of helper thread commonly referred to as thread worker, worker threads, or simply workers.

In [18]:
from time import sleep
from threading import Thread, local, current_thread, main_thread
import threading

In [2]:
# custom function to be executed in a new thread
def task():
    # block for a moment
    sleep(1)
    # report a message
    print('This is from another thread')
    
# protect the entry point
if __name__ == '__main__':
    # create a new thread instance
    thread = Thread(target=task)
    # This does not start the thread immediately, but instead allows the operating system to schedule the function
    # to execute as soon as possible.
    thread.start()
    # wait for the thread to finish.
    print('Waiting for the thread...')
    # explicitly block and wait for the new thread to terminate.
    thread.join()

Waiting for the thread...
This is from another thread


Extending the `thread` class

In [3]:
# custom thread class
class CustomThread(Thread):
    # override the run function
    def run(self):
    # block for a moment
        sleep(1)
        # report a message
        print('This is another thread')

# protect the entry point
if __name__ == '__main__':
    # create the thread. 
    thread = CustomThread()
    # The inherited `start()` method is then called which starts a new thread and executes the content of the `run()`
    # method in the new thread.
    thread.start()
    # wait for the thread to finish
    print('Waiting for the thread to finish')
    thread.join()

Waiting for the thread to finish
This is another thread


**Thread-Local Storage**

It is a mechanism in multithreaded programming that allows data to be stored and accessed in a way that is private to each thread.

Thread-local storage means that every thread in a program has its own personal space to store data. Even if all threads use the same variable names, like `address`, each thread keeps its own separate copy of that variable. This means that one thread can change its address without affecting another thread’s address. It’s like each worker in a team having their own notebook with the same section titles, but each worker writes their own information inside. This is useful when many threads are doing the same task at the same time and need to use the same code, but each one must keep track of its own data without mixing it up with the others.

In [5]:
# custom function executed in a new thread
def task(shared_local):
    # block for a moment to simulate work
    sleep(1)
    # store a private variable on the thread local
    shared_local.value = 33
    # report the stored value
    print(f'Thread stored: {shared_local.value}')
    
# protect the entry point
if __name__ == '__main__':
    # create a shared thread-local instance
    local_storage = local()
    # store a private variable on the thread local
    local_storage.value = 100
    # report the stored value
    print(f'Main stored: {local_storage.value}')
    # create a new thread to run the custom function
    thread = Thread(target=task, args=(local_storage,))
    # start the new thread
    thread.start()
    # wait for the thread to terminate
    thread.join()
    # report the stored value
    print(f'Main sees: {local_storage.value}')

Main stored: 100
Thread stored: 33
Main sees: 100


## Configuring & Interacting with Threads

**1. Configure Thread Name**

Assigning custom names

In [6]:
# protect the entry point
if __name__ == '__main__':
    # create a thread with a custom name
    thread = Thread(name='MyThread')
    # report thread name
    print(thread.name)

MyThread


**2. Configure Daemon Threads**

Threads can be configured to be *daemon* or *daemonic*, that is, they can be configured as backgrounds threads. The main thread can only exit once all non-daemon threads have exited.

In [7]:
# protect the entry point
if __name__ == '__main__':
    # create a daemon thread
    thread = Thread(daemon=True)
    # report if the thread is a daemon
    print(thread.daemon)

True


**3. Query Thread Native Identifier**

Each thread has a unique identifier assigned by the Python interpreter and a unique native thread identifier (TID), assigned by the operating system.

In [8]:
# protect the entry point
if __name__ == '__main__':
    # create a thread
    thread = Thread()
    # report the thread identifier
    print(thread.native_id)
    # start the thread
    thread.start()
    # report the thread identifier
    print(thread.native_id)

None
1944


**4. Query if Thread is Alive**

A `Thread` object can be alive (running) or dead (terminated). If a thread is alive, it means that the `run()` method of the `Thread` object is currently executing. Thus means that before the `start()` method is called and after the `run()` method has completed, the thread will not be alive. 

In [10]:
# protect the entry point
if __name__ == '__main__':
    # create the thread
    thread = Thread()
    # report the thread is alive
    print(thread.is_alive())

False


**5. Get the Current Thread**

We can get a `Thread` object for the thread running the current code.

In [12]:
# protect the entry point
if __name__ == '__main__':
    # get the current thread
    thread = current_thread()
    # report details
    print(thread)

<_MainThread(MainThread, started 18796)>


**6. Get the Main Thread**

In [14]:
# protect the entry point
if __name__ == '__main__':
    # get the main thread
    thread = main_thread()
    # report details
    print(thread)

<_MainThread(MainThread, started 18796)>


**7. Get all active Threads**

`enumerate()` module function that returns an interable of `Thread` objects, one for each running thread.

In [19]:
# custom function to be executed in a new thread
def task():
    # block for a moment
    sleep(1)

# protect the entry point
if __name__ == '__main__':
    # create a number of new threads
    threads = [Thread(target=task) for _ in range(5)]
    # start the new threads
    for thread in threads:
        thread.start()
    # get a list of all running threads
    running_threads = threading.enumerate()
    # report a count of active threads
    print(f'Active Threads: {len(running_threads)}')
    # report each in turn
    for thread in running_threads:
        print(thread)

Active Threads: 11
<_MainThread(MainThread, started 18796)>
<Thread(Thread-6 (_thread_main), started daemon 1984)>
<Heartbeat(Thread-7, started daemon 11148)>
<ControlThread(Thread-5, started daemon 12704)>
<HistorySavingThread(IPythonHistorySavingThread, started 3360)>
<ParentPollerWindows(Thread-4, started daemon 10904)>
<Thread(Thread-20 (task), started 2816)>
<Thread(Thread-21 (task), started 9460)>
<Thread(Thread-22 (task), started 10604)>
<Thread(Thread-23 (task), started 18328)>
<Thread(Thread-24 (task), started 1624)>


**8. Handle Unexpected Exceptions in New Threads**

Ideally, we would like to know when an unexpected exception occurs in new threads so that we might take appropriate action to clean-up any resources and perhaps log the fault.

We can specify a custom exception hook function that will be called whenever a `Thread` fails with an unhandled `Error` or `Exception` using `excepthook`.

In [20]:
# Register a custom function to handle an exception raised in new threads.

# custom exception hook
def custom_hook(args):
    # report the failure
    print(f"Thread failed: {args.exc_value}")
    
# target function that raises an exception
def task():
    # report a message
    print("Working...")
    # block for a moment
    sleep(1)
    # rise an "unexpected" exception
    raise Exception("Something bad happened")
    
# protect the entry point
if __name__ == "__main__":
    # register the exception hook function
    threading.excepthook = custom_hook
    # create a thread
    thread = Thread(target=task)
    # run the thread
    thread.start()
    # wait for the thread to finish
    thread.join()
    # report that the main thread is not dead
    print("Continuing on ...")

Working...
Thread failed: Something bad happened
Continuing on ...
